# Lesson 03 - এজেন্টিক ডিজাইন প্যাটার্নস

এই পাঠে, আমরা কার্যকর AI এজেন্ট তৈরির জন্য তিনটি মৌলিক ডিজাইন প্যাটার্ন অনুসন্ধান করব:

1. **স্পষ্ট এজেন্ট নির্দেশনা** — সুনির্দিষ্ট, ভূমিকা নির্ধারণকারী প্রম্পট তৈরি করা যা এজেন্টের আচরণ নির্দেশ করে  
2. **Pydantic মডেলের সাথে কাঠামোবদ্ধ আউটপুট** — নিশ্চিত করা যে এজেন্টরা পূর্বানুমানযোগ্য, যাচাইকৃত ডেটা ফেরত দিচ্ছে  
3. **একক দায়িত্ব এজেন্টরা** — ফোকাস করা এজেন্ট ডিজাইন করা যারা প্রতিটি এক কাজ ভালো করে করে  

আমরা প্রতিটি প্যাটার্ন প্রয়োগ করব একটি **ভ্রমণ গন্তব্য সুপারিশকারী** দৃশ্যপটে, ক্রমবর্ধমান এমন একটি সিস্টেম তৈরি করব যা গন্তব্য প্রস্তাব করতে পারে, উপলভ্যতা পরীক্ষা করতে পারে, এবং লজিস্টিকস পরিচালনা করতে পারে।


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## প্যাটার্ন ১: স্পষ্ট এজেন্ট নির্দেশাবলী

সবচেয়ে প্রভাবশালী প্যাটার্নও সবচেয়ে সহজ: আপনার এজেন্টের জন্য স্পষ্ট, বিস্তারিত নির্দেশনা লেখা।

ভালো নির্দেশাবলী নির্ধারণ করে:
- **কে** এজেন্ট (ব্যক্তিত্ব এবং স্বর)
- **কি** করতে হবে (পর্যায়ক্রমিক দায়িত্বগুলি)
- **কিভাবে** আচরণ করতে হবে (বাধ্যবাধকতা এবং শৈলী)

নিচে, আমরা একটি ভ্রমণ কনসিয়ার্জ এজেন্ট তৈরি করি যা স্পষ্ট নির্দেশাবলীর মাধ্যমে প্রতিটি প্রতিক্রিয়া গঠন করে।


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## প্যাটার্ন ২: পায়ড্যান্টিক মডেলের সাথে কাঠামোবদ্ধ আউটপুট

ফ্রি-ফর্ম টেক্সট কথোপকথনের জন্য কার্যকর, তবে ডাউনস্ট্রীম সিস্টেমগুলির কাঠামোবদ্ধ ডেটার প্রয়োজন।
**Pydantic মডেলগুলি** কে একটি **টুল ফাংশনের** সাথে জোড়া দিয়ে, আমরা পারি:

- এজেন্টের আউটপুটের জন্য একটি সুনির্দিষ্ট স্কিমা সংজ্ঞায়িত করতে
- স্বয়ংক্রিয়ভাবে প্রতিক্রিয়া যাচাই করতে
- বিশ্বাসযোগ্যভাবে অ্যাপ্লিকেশন লজিকে এজেন্টের ফলাফল সংহত করতে

আমরা এমন একটি টুলও পরিচয় করিয়ে দিই যা গন্তব্যের বিবরণ ফিরিয়ে দেয় যাতে এজেন্ট তার সুপারিশগুলো বাস্তব ডেটার সাথে যুক্ত করতে পারে।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## প্যাটার্ন ৩: একক দায়িত্ব এজেন্ট

জটিল কাজগুলির জন্য একাধিক নিবেদিত এজেন্টের মধ্যে কাজ ভাগ করা উপকারী, যেখানে প্রতিটি এজেন্টের একটি একক দায়িত্ব থাকে:

- একটি **গন্তব্য বিশেষজ্ঞ** যিনি স্থান এবং প্রাপ্যতা সম্পর্কে জানেন
- একটি **লজিস্টিক্স পরিকল্পনাকারী** যিনি ফ্লাইট, হোটেল এবং ভ্রমণসূচি পরিচালনা করেন

এটি সফটওয়্যার প্রকৌশল নীতির *দায়িত্ববিভাজনের* সাথে সাদৃশ্যপূর্ণ — প্রতিটি এজেন্টকে স্বাধীনভাবে পরীক্ষা, রক্ষণাবেক্ষণ, এবং উন্নত করা সহজ হয়।


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## সারাংশ

এই পাঠে আমরা একটি ট্রাভেল রিকমেন্ডার পরিস্থিতিতে তিনটি এজেন্টিক ডিজাইন প্যাটার্ন প্রয়োগ করেছি:

| প্যাটার্ন | মূল ধারণা | সুবিধা |
|---|---|---|
| **স্পষ্ট নির্দেশাবলী** | ব্যক্তিত্ব, দায়িত্ব এবং সীমাবদ্ধতা আগেই নির্ধারণ করুন | সঙ্গতিপূর্ণ, ব্র্যান্ড-মুখী এজেন্ট আচরণ |
| **গঠনমূলক আউটপুট** | উত্তর ফরম্যাট হিসেবে Pydantic মডেল ব্যবহার করুন | যাচাইকৃত, মেশিন-পঠযোগ্য ফলাফল |
| **একক দায়িত্ব** | প্রতিটি এজেন্টকে একটি নির্দিষ্ট কাজ দিন | পরীক্ষা, রক্ষণাবেক্ষণ, এবং সংমিশ্রণ সহজ হয় |

এই প্যাটার্নগুলি প্রাকৃতিকভাবে একত্রিত হয় — আপনি স্পষ্ট নির্দেশাবলী ও গঠনমূলক আউটপুটকে একটি একক-দায়িত্বের এজেন্টের মধ্যে মিলিয়ে শক্তপোক্ত, উৎপাদন-সামর্থ্য সিস্টেম তৈরি করতে পারেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:  
এই ডকুমেন্টটি এআই অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসম্ভব যথার্থতার জন্য চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল ডকুমেন্টটি তার নিজস্ব ভাষায় কর্তৃপক্ষভুক্ত উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
